In [1]:
import pandas as pd
import numpy as np


In [2]:
# Company Shopx File import
df_order = pd.read_excel("Company ShopX - Order Report.xlsx")
df_product = pd.read_excel("Company ShopX - Product Weight.xlsx")
df_zone = pd.read_excel("Company ShopX - Warehouse&Customer Pin Code and Zone details.xlsx")


In [3]:
df_order.info() 

print('---------------------------')
df_product.info()
print('---------------------------')
df_zone.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order ID       400 non-null    int64  
 1   Product Code   400 non-null    object 
 2   Units Ordered  400 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 9.5+ KB
---------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Product Code        66 non-null     object
 1   Product Weight (g)  66 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.2+ KB
---------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124 entries, 0 to 123
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Store H

In [4]:
# courier file File import
df_invoice = pd.read_excel("Courier Company - Invoice.xlsx")
df_rates = pd.read_excel("Courier Company - Rates.xlsx")

In [5]:
df_invoice.info()
print('---------------------------')
df_rates .info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124 entries, 0 to 123
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   AWB Code (Airway Bill Number)   124 non-null    int64  
 1   Order ID                        124 non-null    int64  
 2   Chargeable Weight               124 non-null    float64
 3   Store House Pincode             124 non-null    int64  
 4   Customer Area Code              124 non-null    int64  
 5   Delivery Zone                   124 non-null    object 
 6   Freight Type                    124 non-null    object 
 7   Total Amount (Rs.)              124 non-null    float64
dtypes: float64(2), int64(4), object(2)
memory usage: 7.9+ KB
---------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 

In [6]:
# --------------------------------------------------------------------------------------------------------------------------

In [7]:
# Merge order table and product table

df_merge = pd.merge(df_order, df_product, how='left', on="Product Code")


In [8]:
# Calculate total product weight line by line
df_merge['Weight_by_Company (g)'] = df_merge['Units Ordered'] * df_merge['Product Weight (g)']

In [9]:
# Total order weight at Order ID level

df_company = df_merge.groupby(["Order ID"])["Weight_by_Company (g)"].sum().reset_index()


In [10]:
# -------------------------------------------------------------------------------------

In [11]:
# Convert total weight from gram to KG
df_company["Total weight as per ShopX (KG)"] = df_company["Weight_by_Company (g)"] / 1000

In [12]:
# Merge company order weight with courier invoice
#    Keep invoice on left so AWB stays in final result

df_all = pd.merge(df_invoice, df_company, how="left", on="Order ID")


In [13]:
# Merge zone table on both Store House Pincode and Customer Area Code
df_zone = df_zone.drop_duplicates(subset=["Store House Pincode", "Customer Area Code"])


In [14]:
df_all = pd.merge(
    df_all,
    df_zone[['Store House Pincode', 'Customer Area Code', 'Delivery Zone']],
    on=["Store House Pincode", "Customer Area Code"],
    how="left",
    suffixes=("_by_courier", "_by_company")
)

In [15]:
# Weight slab calculation
#    PDF rule:
#    400 g -> 0.5
#    950 g -> 1.0
#    1000 g -> 1.0
#    2.2 KG -> 2.5

In [16]:
df_all['multipiler_company'] = np.ceil(df_all['Weight_by_Company (g)'] / 500)
df_all['multipiler_courier'] = np.ceil(df_all['Chargeable Weight'] / 0.5)

df_all["slab_by_company"] = np.ceil(df_all["Weight_by_Company (g)"] / 500) * 0.5
df_all["slab_by_courier"] = np.ceil(df_all["Chargeable Weight"] / 0.5) * 0.5


In [17]:
# ------------------------------------------------------------------------------

In [18]:
# Prepare rates table


In [19]:
df_rates = df_rates.T.reset_index().rename(columns={'index': 'types', 0: 'charges'})

In [20]:
# We Keep only valid rate rows like fwd_a_fixed, fwd_a_additional, rto_a_fixed, etc.
df_rates = df_rates[df_rates['types'].str.count('_') == 2].copy()

In [21]:
df_rates['del_zone'] = df_rates['types'].apply(lambda s: s.split('_')[1])
df_rates["charge_type"] = df_rates["types"].apply(lambda s: s.split("_")[0] + "_" + s.split("_")[2])

df_rate_final = pd.pivot(
    data=df_rates,
    index=['del_zone'],
    columns=['charge_type'],
    values=['charges']
)

In [22]:
df_rate_final = df_rate_final.droplevel(level=0, axis='columns').reset_index()
df_rate_final.rename(columns={"del_zone": "Delivery Zone_by_company"}, inplace=True)

In [23]:
# Standardize zone text before merge

In [24]:
df_all["Delivery Zone_by_company"] = df_all["Delivery Zone_by_company"].astype(str).str.lower().str.strip()
df_all["Delivery Zone_by_courier"] = df_all["Delivery Zone_by_courier"].astype(str).str.upper().str.strip()
df_rate_final["Delivery Zone_by_company"] = df_rate_final["Delivery Zone_by_company"].astype(str).str.lower().str.strip()

In [25]:
# Merge rate table into final dataframe
df_final = df_all.merge(df_rate_final, on='Delivery Zone_by_company', how='left')

In [26]:
# Find whether RTO is mentioned or not
df_final.loc[df_final["Freight Type"].str.lower().str.contains("rto", na=False), "rto_mentioned"] = 1
df_final["rto_mentioned"] = df_final["rto_mentioned"].fillna(0)

In [27]:
# -------------------------------------------------------------------------------------------------------

In [28]:
# Calculate expected forward and RTO charges
#    FIX:
#    first 0.5 KG = fixed
#    every extra 0.5 KG = additional

In [29]:
df_final["fwd_charge_by_company"] = (
    df_final["fwd_fixed"] + (df_final["multipiler_company"] - 1) * df_final["fwd_additional"]
)

In [30]:
df_final["rto_charge_by_company"] = (
    df_final["rto_fixed"] + (df_final["multipiler_company"] - 1) * df_final["rto_additional"]
) * df_final["rto_mentioned"]


In [31]:
df_final["total_charge_by_company"] = df_final["fwd_charge_by_company"] + df_final["rto_charge_by_company"]

In [32]:
# Charge status

In [33]:

df_final["variation_in_charge"] = np.where(
    np.isclose(df_final["Total Amount (Rs.)"], df_final["total_charge_by_company"], atol=0.01),
    "Correctly Charged",
    np.where(
        df_final["Total Amount (Rs.)"] > df_final["total_charge_by_company"],
        "Over Charged",
        "Under Charged"
    )
)


In [34]:
# --------------------------------------------------------------------------------------------------------------------

In [35]:
# Amount Difference

In [36]:
df_final['amount_difference'] = df_final['total_charge_by_company'] - df_final['Total Amount (Rs.)']

In [37]:
# ----------------------------------------------------------------------------------------------------

In [38]:
# Rename columns exactly as expected result format

df_final = df_final.rename(columns={
    "AWB Code (Airway Bill Number) ": "AWB Number",
    "Total weight as per ShopX (KG)": "Total weight as per ShopX (KG)",
    "Chargeable Weight": "Total weight as per Courier Company (KG)",
    "slab_by_company": "Weight slab as per ShopX (KG)",
    "slab_by_courier": "Weight slab charged by Courier Company (KG)",
    "Delivery Zone_by_company": "Delivery Zone as per ShopX",
    "Delivery Zone_by_courier": "Delivery Zone charged by Courier Company",
    "total_charge_by_company": "Expected Charge as per ShopX (Rs.)",
    "Total Amount (Rs.)": "Charges Billed by Courier Company (Rs.)",
    "amount_difference": "Difference Between Expected Charges and Billed Charges (Rs.)"
})

In [39]:
# Convert delivery zone as per ShopX back to upper case for final output
df_final["Delivery Zone as per ShopX"] = df_final["Delivery Zone as per ShopX"].astype(str).str.upper()

In [48]:
df_final

,AWB Number,Order ID,Total weight as per Courier Company (KG),Store House Pincode,Customer Area Code,Delivery Zone charged by Courier Company,Freight Type,Charges Billed by Courier Company (Rs.),Weight_by_Company (g),Total weight as per ShopX (KG),...,fwd_additional,fwd_fixed,rto_additional,rto_fixed,rto_mentioned,fwd_charge_by_company,rto_charge_by_company,Expected Charge as per ShopX (Rs.),variation_in_charge,Difference Between Expected Charges and Billed Charges (Rs.)
0,1091117222124,2001806232,1.30,121003,507101,D,Forward charges,135.0,1302.0,1.302,...,44.8,45.4,44.8,41.3,0.0,135.0,0.0,135.0,Correctly Charged,0.000000e+00
1,1091117222194,2001806273,1.00,121003,486886,D,Forward charges,90.2,615.0,0.615,...,44.8,45.4,44.8,41.3,0.0,90.2,0.0,90.2,Correctly Charged,-1.421085e-14
2,1091117222931,2001806408,2.50,121003,532484,D,Forward charges,224.6,2265.0,2.265,...,44.8,45.4,44.8,41.3,0.0,224.6,0.0,224.6,Correctly Charged,0.000000e+00
3,1091117223244,2001806458,1.00,121003,143001,B,Forward charges,61.3,700.0,0.700,...,28.3,33.0,28.3,20.5,0.0,61.3,0.0,61.3,Correctly Charged,0.000000e+00
4,1091117229345,2001807012,0.15,121003,515591,D,Forward charges,45.4,240.0,0.240,...,44.8,45.4,44.8,41.3,0.0,45.4,0.0,45.4,Correctly Charged,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,1091118551656,2001812941,0.73,121003,325207,D,Forward charges,90.2,500.0,0.500,...,28.3,33.0,28.3,20.5,0.0,33.0,0.0,33.0,Over Charged,-5.720000e+01
120,1091117614452,2001809383,0.50,121003,303702,D,Forward and RTO charges,86.7,607.0,0.607,...,28.3,33.0,28.3,20.5,1.0,61.3,48.8,110.1,Under Charged,2.340000e+01
121,1091120922803,2001820978,0.50,121003,313301,D,Forward charges,45.4,515.0,0.515,...,28.3,33.0,28.3,20.5,0.0,61.3,0.0,61.3,Under Charged,1.590000e+01
122,1091121844806,2001811475,0.50,121003,173212,B,Forward charges,33.0,689.0,0.689,...,55.5,56.6,55.5,50.7,0.0,112.1,0.0,112.1,Under Charged,7.910000e+01


In [40]:
# ----------------------------------------------------------

In [41]:
#Create summary table

summary = df_final.groupby('variation_in_charge').agg(
    Count=('Order ID', "count"),
    Amount=('Difference Between Expected Charges and Billed Charges (Rs.)', "sum")
).reset_index()

In [42]:
# Rename summary labels to match business language
summary["variation_in_charge"] = summary["variation_in_charge"].replace({
    "Correctly Charged": "Total Orders - Correctly Charged",
    "Over Charged": "Total Orders - Over Charged",
    "Under Charged": "Total Orders - Under Charged"
})

In [43]:
# Arrange summary rows in required order
summary_order = [
    "Total Orders - Correctly Charged",
    "Total Orders - Over Charged",
    "Total Orders - Under Charged"
]
summary["variation_in_charge"] = pd.Categorical(summary["variation_in_charge"], categories=summary_order, ordered=True)
summary = summary.sort_values("variation_in_charge").reset_index(drop=True)


In [44]:
# --------------------------------------------------------------------------------------------------------------------------

In [45]:
# Final report columns

In [46]:
report_columns = [
    'Order ID',
    'AWB Number',
    'Total weight as per ShopX (KG)',
    'Weight slab as per ShopX (KG)',
    'Total weight as per Courier Company (KG)',
    'Weight slab charged by Courier Company (KG)',
    'Delivery Zone as per ShopX',
    'Delivery Zone charged by Courier Company',
    'Expected Charge as per ShopX (Rs.)',
    'Charges Billed by Courier Company (Rs.)',
    'Difference Between Expected Charges and Billed Charges (Rs.)'
]


In [47]:
# Save final Excel file

writer = pd.ExcelWriter("Result.xlsx", engine='xlsxwriter')
summary.to_excel(writer, sheet_name="Summary", index=False)
df_final[report_columns].round(2).to_excel(writer, sheet_name="Calculations", index=False)
writer.close()